# Phase 1 — BTC/Polymarket Data Foundations

This notebook is the Google Colab execution runbook for the current Phase 1 data workflow. It keeps raw archives in Google Drive, runs the installable package on remote compute, and records compact derived artifacts in the control directory.

Current slice:

```text
raw Binance GZIP
      ↓
manifest + verified UTC coverage
      ↓
audit outputs
      ↓
gap-aware features ─── separate Binance proxy targets
      ↓                              ↓
decision-time inputs       future-window engineering labels
```

The official Chainlink/Polymarket target remains a separate research gate. This notebook does not train a model, backtest a strategy, or trade live.

## Operating rules

Run cells from top to bottom. Rerunning is safe where a cell says it skips or verifies an existing artifact.

- Raw archives are read-only.
- The manifest and coverage map are control artifacts; never overwrite them here.
- June 29 is excluded only from the first derived feature/proxy view because it is source-incomplete and severely under-covered.
- Invalid rows remain in audit outputs and review files. They are not silently filled or deleted.
- Receipt time controls feature eligibility. It does not invalidate an offline target boundary used only to construct a historical label.
- Do not join unresolved proxy targets to features or begin model work before the boundary-recovery gate is accepted.
- Colab is stateless: variables and `/content` files are disposable; Drive checkpoints, manifests, outputs, and reports are the durable state.
- After a runtime reset, rerun the bootstrap cells in order; durable work is then reloaded or skipped rather than recomputed.
- Every long-running cell checkpoints by archive or day and skips verified completed units when rerun.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
from collections import Counter
from datetime import datetime, timedelta, timezone

REPOSITORY = 'https://github.com/matahariramadhan/tradingbot-data.git'
REVISION = '657adfd123767dee6bb4685ab4ee98a2c8314bd2'
PROJECT_DIR = Path('/content/tradingbot_v2')

if not PROJECT_DIR.exists():
    subprocess.run(
        ['git', 'clone', REPOSITORY, str(PROJECT_DIR)],
        check=True,
    )

subprocess.run(
    ['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'],
    check=True,
)
subprocess.run(
    ['git', '-C', str(PROJECT_DIR), 'checkout', '--detach', REVISION],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '.'],
    cwd=PROJECT_DIR,
    check=True,
)

print('repository:', REPOSITORY)
print('revision:', subprocess.check_output(
    ['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'],
).decode().strip())

In [ ]:
import tradingbot_data

assert tradingbot_data.__version__ == '0.7.2', tradingbot_data.__version__
print('package version:', tradingbot_data.__version__)
print(subprocess.check_output(['tradingbot-data', '--help']).decode())

## 1. Mount Drive and define the artifact contract

The paths below are the only project paths this notebook writes. Raw data is never written by the notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RAW_DIR = Path('/content/drive/MyDrive/RecorderBackup/binance-polymarket-recorder')
CONTROL_DIR = Path('/content/drive/MyDrive/tradingbot-data-audit')
MANIFEST = CONTROL_DIR / 'manifest-v2-03abbb7.json'
COVERAGE_MAP = CONTROL_DIR / 'coverage-map-v1.json'
COVERAGE_REPORT = CONTROL_DIR / 'coverage-report-v1.json'
AUDIT_DIR = CONTROL_DIR / 'audit-outputs'
FEATURE_DIR = CONTROL_DIR / 'feature-views'
PROXY_DIR = CONTROL_DIR / 'proxy-targets'
RECOVERED_PROXY_DIR = CONTROL_DIR / 'proxy-targets-recovered-v1'
PROXY_RECOVERY_REPORT = CONTROL_DIR / 'proxy-target-recovery-v1.json'
PROXY_RECOVERED_QUALITY_REPORT = CONTROL_DIR / 'proxy-target-recovered-quality-v1.json'
PROXY_JOIN_AUDIT_DIR = CONTROL_DIR / 'proxy-join-audit-v1'
PROXY_MODEL_DIR = CONTROL_DIR / 'proxy-model-view-v1'
PROXY_JOIN_REPORT = CONTROL_DIR / 'proxy-join-v1.json'
PROXY_REVIEW_REPORT = CONTROL_DIR / 'proxy-model-review-v1.json'
PROXY_EXCLUDED_OUTPUT = CONTROL_DIR / 'proxy-model-excluded-v1.csv'
PROXY_SPLIT_REPORT = CONTROL_DIR / 'proxy-split-v1.json'

EXCLUDED_DAYS = {'2026-06-29'}

assert RAW_DIR.is_dir(), RAW_DIR
assert MANIFEST.is_file(), MANIFEST
assert COVERAGE_MAP.is_file(), COVERAGE_MAP
for directory in (AUDIT_DIR, FEATURE_DIR, PROXY_DIR, RECOVERED_PROXY_DIR, PROXY_JOIN_AUDIT_DIR, PROXY_MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('raw:', RAW_DIR)
print('control:', CONTROL_DIR)
print('manifest:', MANIFEST)
print('coverage map:', COVERAGE_MAP)

## 2. Verify the control artifacts

This section does not recreate the manifest or coverage map. It checks that they describe the expected collection before any derived work runs.

In [ ]:
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
coverage = json.loads(COVERAGE_MAP.read_text(encoding='utf-8'))
records = manifest['records']

assert manifest['manifest_schema_version'] == 2
assert manifest['input_layout'] == 'grouped-gzip'
assert {record.get('input_layout') for record in records} == {'direct_gzip_group'}
assert len(records) == 30
assert len(coverage) == 30

role_counts = Counter(
    role
    for record in records
    for role in record.get('inputs', {})
)
incomplete = [
    (record['group_id'], record.get('missing_input_roles', []))
    for record in records
    if not record.get('input_complete')
]

print('schema:', manifest['manifest_schema_version'])
print('groups:', len(records))
print('roles:', dict(role_counts))
print('incomplete groups:', incomplete)
print('ignored derived files:', len(manifest.get('ignored_files', [])))
print('coverage entries:', len(coverage))

In [ ]:
def parse_utc(value):
    parsed = datetime.fromisoformat(value.replace('Z', '+00:00'))
    assert parsed.tzinfo is not None
    return parsed.astimezone(timezone.utc)

group_ids = {record['group_id'] for record in records}
assert set(coverage) == group_ids

for group_id, value in coverage.items():
    timestamp = parse_utc(value)
    assert timestamp.hour == 0
    assert timestamp.minute == 0
    assert timestamp.second == 0
    assert timestamp.microsecond == 0
    assert value.startswith(group_id + 'T00:00:00')

print('coverage map passed UTC-midnight and group-identity checks')

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

problems = []
for record in records:
    if record.get('status') != 'completed':
        problems.append((record['group_id'], 'status', record.get('status')))
        continue
    output = Path(record.get('audit_output_uri', ''))
    expected = record.get('audit_output_sha256')
    if not output.is_file():
        problems.append((record['group_id'], 'missing_output'))
    elif sha256_file(output) != expected:
        problems.append((record['group_id'], 'checksum_mismatch'))

print('audit statuses:', dict(Counter(record.get('status') for record in records)))
print('audit verification problems:', problems)
assert not problems

In [ ]:
MULTI_DAY_REPORT = CONTROL_DIR / 'multi-day-binance-audit-v1.json'
assert MULTI_DAY_REPORT.is_file(), MULTI_DAY_REPORT
aggregate = json.loads(MULTI_DAY_REPORT.read_text(encoding='utf-8'))
print(json.dumps(aggregate['totals'], indent=2))

## 3. Build the gap-aware feature view

Each eligible day produces 288 five-minute rows. The builder applies interval completion, receipt-time cutoffs, and the consecutive 60-second lookback. Existing outputs are skipped only when their persisted feature report matches the feature-view implementation identity.

In [ ]:
import csv
from scripts.build_binance_feature_view import (
    FEATURE_VIEW_IMPLEMENTATION_VERSION,
    feature_report_is_compatible,
)

FEATURE_COLUMNS = {
    'window_start_utc',
    'decision_time_utc',
    'feature_row_usable',
    'return_1s_valid',
    'return_1m_valid',
    'volatility_1m_valid',
}

def inspect_feature_output(path):
    if not path.is_file():
        return None
    with path.open(newline='', encoding='utf-8') as source:
        reader = csv.DictReader(source)
        rows = list(reader)
    if len(rows) != 288 or not FEATURE_COLUMNS.issubset(reader.fieldnames or []):
        return None
    return {
        'rows': len(rows),
        'usable_rows': sum(row['feature_row_usable'] == 'true' for row in rows),
    }

In [ ]:
FEATURE_REPORT = CONTROL_DIR / 'feature-view-batch-v1.json'
feature_report_matches_implementation = False
if FEATURE_REPORT.is_file():
    previous_feature_report = json.loads(FEATURE_REPORT.read_text(encoding='utf-8'))
    feature_report_matches_implementation = feature_report_is_compatible(previous_feature_report)
feature_results = []

for record in sorted(records, key=lambda item: item['group_id']):
    day = record['group_id']
    if day in EXCLUDED_DAYS:
        feature_results.append({'day': day, 'status': 'excluded_by_policy'})
        continue

    descriptor = record.get('inputs', {}).get('binance_raw')
    coverage_start = coverage.get(day)
    if not descriptor or not coverage_start:
        feature_results.append({'day': day, 'status': 'review', 'reason': 'missing input or coverage'})
        continue

    archive = RAW_DIR / descriptor['relative_path']
    output = FEATURE_DIR / f'{day}.csv'
    existing = inspect_feature_output(output)
    if feature_report_matches_implementation and existing is not None:
        print(day, 'existing valid output; skipping')
        feature_results.append({'day': day, 'status': 'skipped_verified_shape', **existing})
        continue

    temporary = FEATURE_DIR / f'.{day}.csv.tmp'
    print(day, 'processing')
    completed = subprocess.run(
        ['tradingbot-data', 'feature-view', '--archive', str(archive), '--day-start', coverage_start, '--output', str(temporary)],
        text=True, capture_output=True,
    )
    if completed.returncode != 0:
        print(completed.stderr)
        feature_results.append({'day': day, 'status': 'failed', 'error': completed.stderr[-2000:]})
        continue

    temporary.replace(output)
    measured = inspect_feature_output(output)
    if measured is None:
        feature_results.append({'day': day, 'status': 'review', 'reason': 'output verification failed'})
        continue
    print(day, measured)
    feature_results.append({'day': day, 'status': 'completed', **measured})

FEATURE_REPORT.write_text(json.dumps({
    'package_revision': REVISION,
    'package_version': tradingbot_data.__version__,
    'feature_view_implementation_version': FEATURE_VIEW_IMPLEMENTATION_VERSION,
    'excluded_days': sorted(EXCLUDED_DAYS),
    'results': feature_results,
}, indent=2) + '\n', encoding='utf-8')

print('report:', FEATURE_REPORT)
print('statuses:', Counter(item['status'] for item in feature_results))

In [ ]:
INVALID_FEATURE_REPORT = CONTROL_DIR / 'invalid-feature-rows-v1.csv'
feature_report = json.loads(FEATURE_REPORT.read_text(encoding='utf-8'))
invalid_rows = []
feature_flags = Counter()
component_flags = Counter()
total_rows = 0
usable_rows = 0
feature_columns = None

for item in feature_report['results']:
    if item['status'] not in {'completed', 'skipped_verified_shape'}:
        continue
    day = item['day']
    with (FEATURE_DIR / f'{day}.csv').open(newline='', encoding='utf-8') as source:
        reader = csv.DictReader(source)
        feature_columns = reader.fieldnames
        rows = list(reader)
    total_rows += len(rows)
    for row in rows:
        if row['feature_row_usable'] == 'true':
            usable_rows += 1
            continue
        feature_flags[row['feature_quality_flag']] += 1
        invalid_rows.append({'day': day, **row})
        for field in ('return_1s_quality_flag', 'return_1m_quality_flag', 'volatility_1m_quality_flag'):
            if not row[field].startswith('valid_'):
                component_flags[f'{field}={row[field]}'] += 1

with INVALID_FEATURE_REPORT.open('w', newline='', encoding='utf-8') as output:
    writer = csv.DictWriter(output, fieldnames=['day', *feature_columns])
    writer.writeheader()
    writer.writerows(invalid_rows)

print('total rows:', total_rows)
print('usable rows:', usable_rows)
print('invalid rows:', len(invalid_rows))
print('feature flags:', dict(feature_flags))
print('component flags:', dict(component_flags))
print('review file:', INVALID_FEATURE_REPORT)

## 4. Build the separate Binance proxy-target view

This is an engineering label, not the official Polymarket outcome. Its `label_source` must remain `binance_proxy`. Late target-boundary receipts are allowed here because the target is built offline; they must never enter the feature row.

In [ ]:
PROXY_REQUIRED_COLUMNS = {
    'window_start_utc',
    'decision_time_utc',
    'label',
    'label_source',
    'label_definition',
    'target_valid',
    'target_quality_flag',
}

def inspect_proxy_output(path):
    if not path.is_file():
        return None
    with path.open(newline='', encoding='utf-8') as source:
        reader = csv.DictReader(source)
        rows = list(reader)
    if len(rows) != 288 or not PROXY_REQUIRED_COLUMNS.issubset(reader.fieldnames or []):
        return None
    if {row['label_source'] for row in rows} != {'binance_proxy'}:
        return None
    if {row['label_definition'] for row in rows} != {'end_price_gte_start_price'}:
        return None
    return {
        'rows': len(rows),
        'valid_targets': sum(row['target_valid'] == 'true' for row in rows),
        'invalid_targets': sum(row['target_valid'] != 'true' for row in rows),
    }

In [ ]:
PROXY_REPORT = CONTROL_DIR / 'proxy-target-batch-v1.json'
proxy_results = []

for record in sorted(records, key=lambda item: item['group_id']):
    day = record['group_id']
    if day in EXCLUDED_DAYS:
        proxy_results.append({'day': day, 'status': 'excluded_by_policy'})
        continue

    descriptor = record.get('inputs', {}).get('binance_raw')
    coverage_start = coverage.get(day)
    if not descriptor or not coverage_start:
        proxy_results.append({'day': day, 'status': 'review', 'reason': 'missing input or coverage'})
        continue

    start = parse_utc(coverage_start)
    coverage_end = (start + timedelta(days=1)).isoformat(timespec='seconds').replace('+00:00', 'Z')
    archive = RAW_DIR / descriptor['relative_path']
    output = PROXY_DIR / f'{day}.csv'
    existing = inspect_proxy_output(output)
    if existing is not None:
        print(day, 'existing valid output; skipping')
        proxy_results.append({'day': day, 'status': 'skipped_verified_shape', **existing})
        continue

    temporary = PROXY_DIR / f'.{day}.csv.tmp'
    print(day, 'processing')
    completed = subprocess.run(
        ['tradingbot-data', 'proxy-targets', '--archive', str(archive), '--window-start', coverage_start, '--window-end', coverage_end, '--output', str(temporary)],
        text=True, capture_output=True,
    )
    if completed.returncode != 0:
        print(completed.stderr)
        proxy_results.append({'day': day, 'status': 'failed', 'error': completed.stderr[-2000:]})
        continue

    temporary.replace(output)
    measured = inspect_proxy_output(output)
    if measured is None:
        proxy_results.append({'day': day, 'status': 'review', 'reason': 'output verification failed'})
        continue
    diagnostics = json.loads(completed.stderr)
    print(day, diagnostics['target_counters'])
    proxy_results.append({
        'day': day,
        'status': 'completed',
        **measured,
        'target_counters': diagnostics['target_counters'],
    })

PROXY_REPORT.write_text(json.dumps({
    'package_revision': REVISION,
    'package_version': tradingbot_data.__version__,
    'excluded_days': sorted(EXCLUDED_DAYS),
    'results': proxy_results,
}, indent=2) + '\n', encoding='utf-8')

print('report:', PROXY_REPORT)
print('statuses:', Counter(item['status'] for item in proxy_results))

In [ ]:
PROXY_QUALITY_REPORT = CONTROL_DIR / 'proxy-target-quality-v1.json'
proxy_report = json.loads(PROXY_REPORT.read_text(encoding='utf-8'))
missing_end = []
missing_start = []
flag_counts = Counter()

for item in proxy_report['results']:
    if item['status'] not in {'completed', 'skipped_verified_shape'}:
        continue
    day = item['day']
    with (PROXY_DIR / f'{day}.csv').open(newline='', encoding='utf-8') as source:
        rows = list(csv.DictReader(source))
    for row in rows:
        if row['target_valid'] == 'true':
            continue
        flag = row['target_quality_flag']
        flag_counts[flag] += 1
        entry = {'day': day, 'window_start': row['window_start_utc'], 'flag': flag}
        if flag == 'missing_end_boundary':
            missing_end.append(entry)
        elif flag == 'missing_start_boundary':
            missing_start.append(entry)

final_end = [row for row in missing_end if row['window_start'].endswith('23:55:00.000Z')]
intraday_end = [row for row in missing_end if row not in final_end]
quality = {
    'flag_counts': dict(flag_counts),
    'final_23_55_missing_end': final_end,
    'intraday_missing_end': intraday_end,
    'intraday_missing_start': missing_start,
}
PROXY_QUALITY_REPORT.write_text(json.dumps(quality, indent=2) + '\n', encoding='utf-8')

print('flag counts:', dict(flag_counts))
print('final 23:55 missing ends:', len(final_end))
print('intraday missing ends:', len(intraday_end))
print('intraday missing starts:', len(missing_start))
print('report:', PROXY_QUALITY_REPORT)

## 5. Boundary-recovery gate

The repeated final-window missing-end rows may be archive-edge effects. This scan checks whether the expected `23:59:59` boundary appears in any scanned raw archive. It reads raw data but does not modify it.

The scan checkpoints after each completed source archive. If Colab is interrupted, rerun this cell; completed archives are skipped and only the archive being scanned at interruption may be repeated.

Do not join targets to features until the recovery report has been reviewed and a boundary policy has been explicitly accepted.

In [ ]:
from scripts.binance_source import iter_json_records
import os
import tempfile

BOUNDARY_REPORT = CONTROL_DIR / 'proxy-boundary-recovery-v1.json'
BOUNDARY_CHECKPOINT = CONTROL_DIR / 'proxy-boundary-recovery-v1.checkpoint.json'
RESCAN_BOUNDARIES = False
STOP_WHEN_ALL_FOUND = True

def save_boundary_checkpoint(payload):
    with tempfile.NamedTemporaryFile(
        mode='w', encoding='utf-8', dir=CONTROL_DIR,
        prefix='.proxy-boundary-', suffix='.tmp', delete=False,
    ) as temporary:
        json.dump(payload, temporary, indent=2)
        temporary.write('\n')
        temporary_path = temporary.name
    os.replace(temporary_path, BOUNDARY_CHECKPOINT)

def save_boundary_report(payload):
    with tempfile.NamedTemporaryFile(
        mode='w', encoding='utf-8', dir=CONTROL_DIR,
        prefix='.proxy-boundary-report-', suffix='.tmp', delete=False,
    ) as temporary:
        json.dump(payload, temporary, indent=2)
        temporary.write('\n')
        temporary_path = temporary.name
    os.replace(temporary_path, BOUNDARY_REPORT)

if BOUNDARY_REPORT.is_file() and not RESCAN_BOUNDARIES:
    print('existing boundary report; skipping scan:', BOUNDARY_REPORT)
else:
    wanted = {}
    for record in records:
        day = record['group_id']
        if day in EXCLUDED_DAYS:
            continue
        start = parse_utc(f'{day}T00:00:00Z')
        boundary = start + timedelta(hours=23, minutes=59, seconds=59)
        wanted[int(boundary.timestamp() * 1000)] = day

    wanted_as_text = {str(key): value for key, value in wanted.items()}
    state = None
    if BOUNDARY_CHECKPOINT.is_file() and not RESCAN_BOUNDARIES:
        candidate = json.loads(BOUNDARY_CHECKPOINT.read_text(encoding='utf-8'))
        if (
            candidate.get('checkpoint_schema_version') == 1
            and candidate.get('wanted') == wanted_as_text
            and set(candidate.get('found', {})) == set(wanted_as_text)
        ):
            state = candidate
            print('resuming checkpoint:', BOUNDARY_CHECKPOINT)

    if state is None:
        state = {
            'checkpoint_schema_version': 1,
            'wanted': wanted_as_text,
            'completed_sources': [],
            'found': {str(key): [] for key in wanted},
        }

    completed_sources = set(state['completed_sources'])
    found = {int(key): value for key, value in state['found'].items()}
    scan_complete = False

    try:
        for record in records:
            source_group = record['group_id']
            if source_group in completed_sources:
                continue
            descriptor = record.get('inputs', {}).get('binance_raw')
            if not descriptor:
                completed_sources.add(source_group)
                save_boundary_checkpoint({
                    'checkpoint_schema_version': 1,
                    'wanted': wanted_as_text,
                    'completed_sources': sorted(completed_sources),
                    'found': {str(key): value for key, value in found.items()},
                })
                continue
            archive = RAW_DIR / descriptor['relative_path']
            print('scanning:', source_group)
            for payload in iter_json_records(archive):
                if payload is None or payload.get('stream') != 'btcusdt@kline_1s':
                    continue
                kline = payload.get('raw_event', {}).get('k', {})
                if kline.get('x') is not True:
                    continue
                start_ms = int(kline['t'])
                if start_ms in wanted:
                    found[start_ms].append({
                        'source_group': source_group,
                        'source_file': descriptor['relative_path'],
                        'close': kline['c'],
                        'received_at_utc': payload['received_at_utc'],
                    })
            completed_sources.add(source_group)
            save_boundary_checkpoint({
                'checkpoint_schema_version': 1,
                'wanted': wanted_as_text,
                'completed_sources': sorted(completed_sources),
                'found': {str(key): value for key, value in found.items()},
            })
            print('checkpoint saved; completed sources:', len(completed_sources))
            if STOP_WHEN_ALL_FOUND and all(found.values()):
                print('all boundaries found; stopping early')
                scan_complete = True
                break
        else:
            scan_complete = True
    except KeyboardInterrupt:
        print('interrupted safely; rerun this cell to resume from the checkpoint')

    boundary_results = [
        {
            'target_day': target_day,
            'boundary_start_ms': boundary_ms,
            'sources': found[boundary_ms],
            'recoverable': bool(found[boundary_ms]),
        }
        for boundary_ms, target_day in sorted((key, value) for key, value in wanted.items())
    ]
    if scan_complete:
        assert len(boundary_results) == len(wanted)
        assert {item['target_day'] for item in boundary_results} == set(wanted.values())
        save_boundary_report(boundary_results)
        print('complete boundary report:', BOUNDARY_REPORT)
    else:
        print('final report not written; checkpoint is incomplete:', BOUNDARY_CHECKPOINT)

In [ ]:
boundary_results = json.loads(BOUNDARY_REPORT.read_text(encoding='utf-8'))
recoverable = [item for item in boundary_results if item['recoverable']]
unrecoverable = [item for item in boundary_results if not item['recoverable']]
print('requested boundaries:', len(boundary_results))
print('recoverable boundaries:', len(recoverable))
print('unrecoverable boundaries:', len(unrecoverable))
for item in unrecoverable:
    print('UNRECOVERABLE:', item['target_day'])

## 6. Build and verify the separate recovered proxy view

This command consumes the completed cross-archive boundary report. It writes to a new Drive directory, checkpoints one day at a time, and never overwrites the original proxy-target CSVs.

The command requires the pinned `0.5.0` package revision from the setup cell.

In [ ]:
recovery_command = [
    'tradingbot-data', 'proxy-recover',
    '--input-dir', str(PROXY_DIR),
    '--output-dir', str(RECOVERED_PROXY_DIR),
    '--boundary-report', str(BOUNDARY_REPORT),
    '--output-report', str(PROXY_RECOVERY_REPORT),
]
completed = subprocess.run(recovery_command, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f'proxy recovery failed with exit code {completed.returncode}')

recovery = json.loads(PROXY_RECOVERY_REPORT.read_text(encoding='utf-8'))
assert recovery['status'] == 'completed', recovery['status']
assert recovery['totals']['recovered_rows'] == len(recoverable)
assert recovery['totals']['unused_recovered_boundaries'] == 0
print('recovered rows:', recovery['totals']['recovered_rows'])
print('recovered output directory:', RECOVERED_PROXY_DIR)

In [ ]:
import csv
from collections import Counter

recovery = json.loads(PROXY_RECOVERY_REPORT.read_text(encoding='utf-8'))
quality_by_day = []
total_rows = 0
valid_rows = 0
recovered_rows = 0
quality_flags = Counter()

for day in sorted(recovery['days']):
    output = RECOVERED_PROXY_DIR / f'{day}.csv'
    assert output.is_file(), output
    with output.open(newline='', encoding='utf-8') as source:
        rows = list(csv.DictReader(source))
    assert len(rows) == 288, (day, len(rows))
    day_valid = sum(row['target_valid'] == 'true' for row in rows)
    day_recovered = sum(row['target_quality_flag'].startswith('valid_target_recovered_end_boundary') for row in rows)
    total_rows += len(rows)
    valid_rows += day_valid
    recovered_rows += day_recovered
    quality_flags.update(row['target_quality_flag'] for row in rows if row['target_valid'] != 'true')
    quality_by_day.append({'day': day, 'rows': len(rows), 'valid_rows': day_valid, 'recovered_rows': day_recovered})

recovered_quality = {
    'recovery_report': str(PROXY_RECOVERY_REPORT),
    'recovered_target_directory': str(RECOVERED_PROXY_DIR),
    'total_rows': total_rows,
    'valid_rows': valid_rows,
    'invalid_rows': total_rows - valid_rows,
    'recovered_rows': recovered_rows,
    'invalid_quality_flags': dict(quality_flags),
    'days': quality_by_day,
}
temporary = PROXY_RECOVERED_QUALITY_REPORT.with_name(f'.{PROXY_RECOVERED_QUALITY_REPORT.name}.tmp')
temporary.write_text(json.dumps(recovered_quality, indent=2) + '\n', encoding='utf-8')
temporary.replace(PROXY_RECOVERED_QUALITY_REPORT)
assert recovered_rows == recovery['totals']['recovered_rows']
print(json.dumps(recovered_quality, indent=2))
print('quality report:', PROXY_RECOVERED_QUALITY_REPORT)

## 7. Build and verify the model-ready proxy join

This step matches the feature and recovered-target views by `window_start_utc`. Duplicate keys stop the join. The audit join preserves invalid and unmatched rows; the model-ready view keeps only rows where both the feature row and target are valid. Its initial model features are `return_1s`, `return_1m`, and `volatility_1m`; target prices and target receipt metadata are excluded from the model columns.

In [ ]:
join_command = [
    'tradingbot-data', 'proxy-join',
    '--feature-dir', str(FEATURE_DIR),
    '--target-dir', str(RECOVERED_PROXY_DIR),
    '--audit-output-dir', str(PROXY_JOIN_AUDIT_DIR),
    '--model-output-dir', str(PROXY_MODEL_DIR),
    '--output-report', str(PROXY_JOIN_REPORT),
]
completed = subprocess.run(join_command, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f'proxy join failed with exit code {completed.returncode}')

join_report = json.loads(PROXY_JOIN_REPORT.read_text(encoding='utf-8'))
assert join_report['status'] == 'completed', join_report.get('error', join_report['status'])
assert join_report['totals']['input_days'] == 29
print('join report:', PROXY_JOIN_REPORT)

In [ ]:
join_report = json.loads(PROXY_JOIN_REPORT.read_text(encoding='utf-8'))
audit_rows_total = 0
model_rows_total = 0
eligible_rows_total = 0
model_columns = join_report['model_columns']
for day, result in sorted(join_report['days'].items()):
    assert result['status'] == 'completed', (day, result)
    audit_path = PROXY_JOIN_AUDIT_DIR / f'{day}.csv'
    model_path = PROXY_MODEL_DIR / f'{day}.csv'
    assert audit_path.is_file(), audit_path
    assert model_path.is_file(), model_path
    with audit_path.open(newline='', encoding='utf-8') as source:
        audit_rows = list(csv.DictReader(source))
    with model_path.open(newline='', encoding='utf-8') as source:
        model_reader = csv.DictReader(source)
        model_rows = list(model_reader)
    assert len(audit_rows) == 288, (day, len(audit_rows))
    assert model_reader.fieldnames == model_columns
    assert all(row['label_source'] == 'binance_proxy' for row in model_rows)
    eligible = sum(row['eligible_for_model'] == 'true' for row in audit_rows)
    assert eligible == len(model_rows), (day, eligible, len(model_rows))
    audit_rows_total += len(audit_rows)
    model_rows_total += len(model_rows)
    eligible_rows_total += eligible

assert audit_rows_total == 29 * 288
assert model_rows_total == eligible_rows_total
assert 'target_proxy_end_price' not in model_columns
assert 'target_target_available_at_utc' not in model_columns
print('audit rows:', audit_rows_total)
print('model-ready rows:', model_rows_total)
print('model columns:', model_columns)
print('audit join directory:', PROXY_JOIN_AUDIT_DIR)
print('model-ready directory:', PROXY_MODEL_DIR)

## 8. Review proxy model-view quality

This is a dataset-quality gate, not model training. It checks label balance, finite numeric features, canonical chronological keys, proxy-label provenance, and the reasons for excluded rows.

In [ ]:
review_command = [
    'tradingbot-data', 'proxy-review',
    '--audit-dir', str(PROXY_JOIN_AUDIT_DIR),
    '--model-dir', str(PROXY_MODEL_DIR),
    '--output-report', str(PROXY_REVIEW_REPORT),
    '--excluded-output', str(PROXY_EXCLUDED_OUTPUT),
]
completed = subprocess.run(review_command, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f'proxy model review failed with exit code {completed.returncode}')

review_report = json.loads(PROXY_REVIEW_REPORT.read_text(encoding='utf-8'))
assert review_report['status'] == 'completed', review_report.get('error', review_report['status'])
assert review_report['totals']['audit_rows'] == join_report['totals']['audit_rows']
assert review_report['totals']['model_rows'] == join_report['totals']['eligible_rows']
assert review_report['totals']['excluded_rows'] == (
    review_report['totals']['audit_rows'] - review_report['totals']['model_rows']
)
print('review report:', PROXY_REVIEW_REPORT)
print('excluded-row review:', PROXY_EXCLUDED_OUTPUT)
print('label counts:', review_report['totals']['label_counts'])
print('feature stats:', review_report['feature_stats'])

## 9. Build the chronological proxy split

This creates a durable Drive report for the accepted 23-day training / 6-day evaluation split. It does not copy rows or train a model.

In [ ]:
split_command = [
    'tradingbot-data', 'proxy-split',
    '--model-dir', str(PROXY_MODEL_DIR),
    '--review-report', str(PROXY_REVIEW_REPORT),
    '--output-report', str(PROXY_SPLIT_REPORT),
    '--train-day-count', '23',
]
completed = subprocess.run(split_command, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f'proxy split failed with exit code {completed.returncode}')

split_report = json.loads(PROXY_SPLIT_REPORT.read_text(encoding='utf-8'))
eligible_days = sorted(
    record['group_id'] for record in records if record['group_id'] not in EXCLUDED_DAYS
)
assert split_report['status'] == 'completed', split_report.get('error', split_report['status'])
assert split_report['train_days'] == eligible_days[:23]
assert split_report['evaluation_days'] == eligible_days[23:]
assert split_report['verification']['model_rows_match_review'] is True
assert split_report['verification']['train_evaluation_overlap_keys'] == 0
assert split_report['totals']['model_rows'] == review_report['totals']['model_rows']
assert split_report['totals']['train_rows'] + split_report['totals']['evaluation_rows'] == split_report['totals']['model_rows']
print('split report:', PROXY_SPLIT_REPORT)
print('training days:', split_report['train_days'])
print('evaluation days:', split_report['evaluation_days'])
print('row totals:', split_report['totals'])
print('train/evaluation overlap keys:', split_report['verification']['train_evaluation_overlap_keys'])

## 10. Current stopping point

At this point the notebook has verified the remote audit, built and reviewed the feature view, original proxy view, recovered proxy view, audit join, leakage-safe model-ready proxy view, proxy dataset quality report, and chronological proxy split report.

Do not train yet. The split is an engineering/proxy baseline only. Official Chainlink labels and the separate Polymarket-faithful task remain unresolved.